# Iris Classification with MLflow and SageMaker Studio

This notebook demonstrates how to use MLflow for experiment tracking and model registry with a simple iris classification problem.

## Prerequisites
- SageMaker Studio with MLflow tracking server configured
- Required packages installed (see requirements.txt)

In [ ]:
# Install requirements if not already installed
!pip install -r ../requirements.txt

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

# Add src to path
sys.path.append('../src')

from data.data_loader import load_iris_dataset, preprocess_data
from models.train import run_experiment, run_all_models
from models.evaluate import ExperimentAnalyzer, register_champion_model
from utils.mlflow_utils import MLflowTracker
from config.mlflow_config import MLFLOW_CONFIG

# Set style
plt.style.use('default')
sns.set_palette("husl")

## 1. Setup MLflow Configuration

Configure MLflow to work with SageMaker Studio's tracking server.

In [ ]:
# MLflow configuration
EXPERIMENT_NAME = "iris_sagemaker_studio_demo"

# Set MLflow tracking URI (use SageMaker MLflow tracking server if available)
# For local development, this will use local tracking
TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', MLFLOW_CONFIG['tracking_uri'])
mlflow.set_tracking_uri(TRACKING_URI)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment Name: {EXPERIMENT_NAME}")

## 2. Explore the Dataset

Let's load and explore the iris dataset.

In [ ]:
# Load dataset
X, y = load_iris_dataset()

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"Target classes: {y.unique()}")
print(f"Class distribution:\n{y.value_counts()}")

# Display first few rows
display(X.head())
display(y.head())

In [ ]:
# Visualize the dataset
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Iris Dataset Exploration', fontsize=16)

# Pairwise feature relationships
features = X.columns[:4]
for i, (ax, feature) in enumerate(zip(axes.flat, features)):
    for class_name in y.unique():
        mask = y == class_name
        ax.hist(X.loc[mask, feature], alpha=0.7, label=class_name, bins=15)
    ax.set_title(f'Distribution of {feature}')
    ax.set_xlabel(feature)
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Prepare the data for training.

In [ ]:
# Preprocess data
data = preprocess_data(X, y, test_size=0.2, random_state=42)

X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Features scaled: {data['scaler'] is not None}")

## 4. Train Individual Models

Train individual models and track them with MLflow.

In [ ]:
# Train a single model (example: Random Forest)
print("Training Random Forest model...")
rf_run_id = run_experiment(
    model_name='random_forest',
    experiment_name=EXPERIMENT_NAME,
    tracking_uri=TRACKING_URI
)

print(f"Random Forest Run ID: {rf_run_id}")

## 5. Compare Multiple Models

Train multiple models and compare their performance.

In [ ]:
# Train all available models
print("Training all models for comparison...")
run_ids = run_all_models(
    experiment_name=EXPERIMENT_NAME,
    tracking_uri=TRACKING_URI
)

print("\nTraining completed!")
for model_name, run_id in run_ids.items():
    status = "✅ SUCCESS" if run_id else "❌ FAILED"
    print(f"{model_name}: {status}")

## 6. Analyze and Compare Results

Use the experiment analyzer to compare model performance.

In [ ]:
# Initialize experiment analyzer
analyzer = ExperimentAnalyzer(EXPERIMENT_NAME, TRACKING_URI)

# Print summary
analyzer.print_summary()

In [ ]:
# Get detailed comparison
comparison_df = analyzer.compare_models()
print("Detailed Model Comparison:")
display(comparison_df.round(4))

In [ ]:
# Plot model comparison
analyzer.plot_model_comparison(figsize=(15, 10))

## 7. Register Best Model

Register the best performing model to MLflow Model Registry.

In [ ]:
# Register the best model
model_name = "iris_champion_model_demo"

model_uri = register_champion_model(
    experiment_name=EXPERIMENT_NAME,
    model_name=model_name,
    metric="test_accuracy",
    tracking_uri=TRACKING_URI
)

print(f"Model registered: {model_name}")
print(f"Model URI: {model_uri}")

## 8. Load and Test Registered Model

Load the registered model and test it on new data.

In [ ]:
# Load the registered model
model_name = "iris_champion_model_demo"
model_version = 1  # or "latest"

model_uri = f"models:/{model_name}/{model_version}"
loaded_model = mlflow.sklearn.load_model(model_uri)

print(f"Loaded model: {model_name} version {model_version}")
print(f"Model type: {type(loaded_model).__name__}")

In [ ]:
# Test the model on test data
predictions = loaded_model.predict(X_test)
probabilities = loaded_model.predict_proba(X_test)

from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, predictions)
print(f"Test Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

In [ ]:
# Example predictions on new data points
print("Example predictions:")
for i in range(min(5, len(X_test))):
    sample = X_test.iloc[i:i+1]
    prediction = loaded_model.predict(sample)[0]
    probability = loaded_model.predict_proba(sample)[0]
    actual = y_test.iloc[i]
    
    print(f"Sample {i+1}:")
    print(f"  Predicted: {prediction} (confidence: {max(probability):.3f})")
    print(f"  Actual: {actual}")
    print(f"  Correct: {'✅' if prediction == actual else '❌'}")
    print()

## 9. Model Cards and Documentation

Create model documentation for the registered model.

In [ ]:
# Get model information for documentation
best_model_info = analyzer.get_best_model()

model_card = f"""
# Model Card: Iris Classification Champion Model

## Model Overview
- **Model Name**: {model_name}
- **Algorithm**: {best_model_info['model_name']}
- **Version**: {model_version}
- **Training Date**: {best_model_info['start_time']}
- **MLflow Run ID**: {best_model_info['run_id']}

## Performance Metrics
- **Test Accuracy**: {best_model_info['best_test_accuracy']:.4f}

## Dataset
- **Dataset**: Iris Flower Classification
- **Features**: Sepal length, Sepal width, Petal length, Petal width
- **Classes**: Setosa, Versicolor, Virginica
- **Training Samples**: {len(X_train)}
- **Test Samples**: {len(X_test)}

## Usage
```python
import mlflow.sklearn
model = mlflow.sklearn.load_model("models:/{model_name}/{model_version}")
predictions = model.predict(your_data)
```

## Deployment Notes
- Model is ready for deployment to SageMaker endpoints
- Requires feature scaling (scaler included in model artifacts)
- Input features must match training data schema
"""

print(model_card)

## 10. Next Steps

This notebook demonstrated:
1. ✅ Data loading and preprocessing
2. ✅ Multiple model training with MLflow tracking
3. ✅ Model comparison and evaluation
4. ✅ Model registration in MLflow Model Registry
5. ✅ Model loading and testing
6. ✅ Model documentation

### For SageMaker Studio Integration:
1. **Setup MLflow Tracking Server** in SageMaker Studio
2. **Configure Environment Variables** for tracking URI and S3 storage
3. **Deploy Models** using SageMaker Model Registry integration
4. **Monitor Models** using SageMaker Model Monitor

### For Production Use:
1. **Hyperparameter Tuning** using SageMaker Hyperparameter Tuning Jobs
2. **Model Deployment** to SageMaker Endpoints
3. **A/B Testing** with multiple model versions
4. **Automated Retraining** pipelines with SageMaker Pipelines